# Restartable RLMF Colab Orchestrator

This notebook delegates data preparation, training, checkpoint sealing, archive validation, and runtime version validation to the checked-in Task 7 package and CLI. Local scratch contains all live artifacts; optional Drive storage contains only immutable checkpoint archives.


## Select one run mode

- `smoke`: the registered smoke config and selected seed.
- `pilot`: the confirmatory config's explicit 25-step infrastructure pilot. Its RL checkpoints remain intentionally incomplete and are recorded only as infrastructure artifacts.
- `confirmatory`: the unchanged confirmatory config and one selected registered seed.

Set `RUN_MODE`, `PROJECT_COMMIT`, and optionally `PROJECT_SOURCE`, `SELECTED_SEED`, and `USE_DRIVE=1` before running from the top. `PROJECT_COMMIT` must be a full Git SHA. Evaluation is unavailable until Task 10 provides the locked audit boundary and real CLI commands.


In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

RUN_MODE = os.environ.get("RUN_MODE", "smoke")
USE_DRIVE = os.environ.get("USE_DRIVE", "0") == "1"
PROJECT_COMMIT = os.environ.get("PROJECT_COMMIT", "")
PROJECT_SOURCE = os.environ.get("PROJECT_SOURCE", "/content/project-upload")
PROJECT_ROOT = Path("/content/metacognitive-feature-flow")
MODE_CONFIGS = {
    "smoke": "configs/rlmf_qwen06b_smoke.json",
    "pilot": "configs/rlmf_qwen06b_confirmatory.json",
    "confirmatory": "configs/rlmf_qwen06b_confirmatory.json",
}
if RUN_MODE not in MODE_CONFIGS:
    raise ValueError(f"RUN_MODE must be one of {sorted(MODE_CONFIGS)}")
CONFIG_PATH = MODE_CONFIGS[RUN_MODE]
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise ValueError("PROJECT_COMMIT must be an exact lowercase 40-character Git SHA")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", PROJECT_SOURCE, str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / ".git").exists():
    raise RuntimeError("PROJECT_ROOT must be a Git repository")
subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", PROJECT_COMMIT], check=True)
resolved_commit = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True).strip()
if resolved_commit != PROJECT_COMMIT:
    raise RuntimeError("checked-out commit does not match PROJECT_COMMIT")
dirty_paths = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"], text=True).strip()
if dirty_paths:
    raise RuntimeError(f"project checkout is dirty: {dirty_paths}")
os.chdir(PROJECT_ROOT)
print(json.dumps({"project_commit": resolved_commit, "run_mode": RUN_MODE}, sort_keys=True))


## Install and verify the frozen runtime

The exact direct roots and test/build tooling are installed under the checked-in Python 3.12 Linux constraints, then the project is installed editable with dependency resolution disabled. Task 7 validates the scientific runtime versions.


In [ ]:
TOOLING_REQUIREMENTS = [
    "pip==26.1.2",
    "pytest==9.1.1",
    "setuptools==83.0.0",
    "wheel==0.47.0",
]
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--constraint",
        "requirements-rlmf-colab.constraints.txt",
        "--requirement",
        "requirements-rlmf-colab.txt",
        *TOOLING_REQUIREMENTS,
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)

from trajectory_extractor.rlmf_artifacts import RLMFArtifactStore
from trajectory_extractor.rlmf_training import (
    import_checkpoint,
    latest_verified_checkpoint,
    validate_runtime_versions,
)
from trajectory_extractor.rlmf_types import CheckpointRecord, RLMFConfig

runtime_versions = validate_runtime_versions()
print(json.dumps({"runtime_versions": runtime_versions}, sort_keys=True))


## Configure isolated storage

All working state stays under `/content/rlmf-scratch`. Drive is mounted only when `USE_DRIVE=1` and receives only content-bound `.tar` checkpoint archives.


In [ ]:
config = RLMFConfig.from_json(CONFIG_PATH)
SELECTED_SEED = int(os.environ.get("SELECTED_SEED", "11"))
if SELECTED_SEED not in config.seeds:
    raise ValueError("SELECTED_SEED is not registered in the selected config")
expected_profile = "smoke" if RUN_MODE == "smoke" else "confirmatory"
if config.profile != expected_profile:
    raise RuntimeError("run mode and checked-in config profile do not match")
if RUN_MODE == "pilot" and SELECTED_SEED != config.seeds[0]:
    raise ValueError("infrastructure pilot requires the first registered seed")

SCRATCH_ROOT = Path("/content/rlmf-scratch")
ARTIFACT_ROOT = SCRATCH_ROOT / "artifacts"
LOCAL_EXPORT_ROOT = SCRATCH_ROOT / "exports" / PROJECT_COMMIT / RUN_MODE
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_EXPORT_ROOT = None
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_EXPORT_ROOT = Path("/content/drive/MyDrive/rlmf-checkpoints") / PROJECT_COMMIT / RUN_MODE
    DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
store = RLMFArtifactStore(ARTIFACT_ROOT)
print(
    json.dumps(
        {
            "artifact_root": str(ARTIFACT_ROOT),
            "config": CONFIG_PATH,
            "drive_enabled": USE_DRIVE,
            "seed": SELECTED_SEED,
        },
        sort_keys=True,
    )
)


## Enforce the Colab GPU gate

Execution stops before data or model work unless CUDA is available and the selected device reports at least 14 GB total VRAM.


In [ ]:
import torch

MIN_GPU_MEMORY_GB = 14
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
gpu_properties = torch.cuda.get_device_properties(0)
gpu_memory_gb = gpu_properties.total_memory / (1024 ** 3)
if gpu_memory_gb < MIN_GPU_MEMORY_GB:
    raise RuntimeError(f"GPU has {gpu_memory_gb:.2f} GB; at least {MIN_GPU_MEMORY_GB} GB is required")
print(
    json.dumps(
        {
            "gpu_name": gpu_properties.name,
            "gpu_memory_gb": round(gpu_memory_gb, 3),
            "minimum_gpu_memory_gb": MIN_GPU_MEMORY_GB,
        },
        sort_keys=True,
    )
)


## Verify upstream manifests and sealed data

The preregistration tests validate vendored upstream hashes. Data preparation remains a CLI operation, and every path must end at the sealed Task 7 endpoint verifier.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_rlmf_preregistration.py", "-q"],
    check=True,
)
prepare_marker = (
    ARTIFACT_ROOT
    / "runs"
    / "rlmf"
    / config.study_id
    / "endpoints"
    / "prepare-data.complete.json"
)
if not prepare_marker.exists():
    subprocess.run(
        [
            "feature-dynamics",
            "rlmf-prepare-data",
            "--config",
            CONFIG_PATH,
            "--root",
            str(ARTIFACT_ROOT),
        ],
        check=True,
    )
data_manifest = store.verify_endpoint(config.study_id, "prepare-data")
print(
    json.dumps(
        {
            "data_endpoint": data_manifest["endpoint"],
            "data_parent_hashes": data_manifest["parent_hashes"],
        },
        sort_keys=True,
    )
)


## Define restart and immutable archive operations

These helpers contain storage orchestration only. Task 7 verifies live checkpoint directories and fully binds every imported archive to config, stage, arm, seed, and pre-SFT parent.


In [ ]:
checkpoint_root = ARTIFACT_ROOT / "runs" / "rlmf" / config.study_id / "checkpoints"


def load_checkpoint_records(stage):
    records = []
    for manifest_path in sorted(checkpoint_root.glob("*/checkpoint.json")):
        record = CheckpointRecord.from_record(json.loads(manifest_path.read_text()))
        if (
            record.stage != stage["record_stage"]
            or record.arm != stage["record_arm"]
            or record.seed != stage["seed"]
        ):
            continue
        verified_path = latest_verified_checkpoint(Path(record.path))
        if verified_path is None or verified_path.resolve() != Path(record.path).resolve():
            raise RuntimeError(f"sealed checkpoint verification failed: {record.path}")
        records.append(record)
    return records


def verify_checkpoint_archive(archive_path, stage, expected_pre_sft_hash):
    with tempfile.TemporaryDirectory(dir=SCRATCH_ROOT) as verify_root:
        verified = import_checkpoint(
            RLMFArtifactStore(Path(verify_root)),
            archive_path,
            config=config,
            expected_stage=stage["record_stage"],
            expected_arm=stage["record_arm"],
            expected_seed=stage["seed"],
            expected_pre_sft_hash=expected_pre_sft_hash,
        )
    return verified


def publish_checkpoint_archive(record, stage, expected_pre_sft_hash):
    verified_path = latest_verified_checkpoint(Path(record.path))
    if verified_path is None or verified_path.resolve() != Path(record.path).resolve():
        raise RuntimeError(f"checkpoint is not sealed: {record.path}")
    archive_scope = stage["seed"] if stage["seed"] is not None else "shared"
    archive_name = (
        f"{config.study_id}-{stage['name']}-{archive_scope}-"
        f"step-{record.global_step:06d}-{record.checkpoint_hash}.tar"
    )
    local_target = LOCAL_EXPORT_ROOT / archive_name
    if local_target.exists():
        verified = verify_checkpoint_archive(local_target, stage, expected_pre_sft_hash)
        if verified.checkpoint_hash != record.checkpoint_hash:
            raise FileExistsError(f"existing local archive hash mismatch: {local_target}")
    else:
        with tempfile.TemporaryDirectory(dir=SCRATCH_ROOT) as export_root:
            temporary_archive = Path(export_root) / archive_name
            subprocess.run(
                [
                    "feature-dynamics",
                    "rlmf-export-checkpoint",
                    "--artifact-root",
                    str(ARTIFACT_ROOT),
                    "--checkpoint",
                    str(record.path),
                    "--output",
                    str(temporary_archive),
                ],
                check=True,
            )
            with temporary_archive.open("rb") as source, local_target.open("xb") as output:
                shutil.copyfileobj(source, output)
                output.flush()
                os.fsync(output.fileno())
        verified = verify_checkpoint_archive(local_target, stage, expected_pre_sft_hash)
        if verified.checkpoint_hash != record.checkpoint_hash:
            raise RuntimeError(f"new local archive hash mismatch: {local_target}")

    published_target = local_target
    if USE_DRIVE:
        drive_target = DRIVE_EXPORT_ROOT / archive_name
        if drive_target.exists():
            verified = verify_checkpoint_archive(drive_target, stage, expected_pre_sft_hash)
            if verified.checkpoint_hash != record.checkpoint_hash:
                raise FileExistsError(f"existing Drive archive hash mismatch: {drive_target}")
        else:
            with local_target.open("rb") as source, drive_target.open("xb") as output:
                shutil.copyfileobj(source, output)
                output.flush()
                os.fsync(output.fileno())
            verified = verify_checkpoint_archive(drive_target, stage, expected_pre_sft_hash)
            if verified.checkpoint_hash != record.checkpoint_hash:
                raise RuntimeError(f"new Drive archive hash mismatch: {drive_target}")
        published_target = drive_target
    return str(published_target)


def restore_stage_archives(stage, expected_pre_sft_hash):
    if not USE_DRIVE:
        return []
    archive_scope = stage["seed"] if stage["seed"] is not None else "shared"
    archive_prefix = f"{config.study_id}-{stage['name']}-{archive_scope}-step-"
    verified_archives = []
    for archive_path in sorted(DRIVE_EXPORT_ROOT.glob(f"{archive_prefix}*.tar")):
        archived = verify_checkpoint_archive(archive_path, stage, expected_pre_sft_hash)
        verified_archives.append((archive_path, archived))

    selected_by_step = {}
    for archive_path, archived in verified_archives:
        selected = selected_by_step.get(archived.global_step)
        if selected is None or (archived.completed and not selected[1].completed):
            selected_by_step[archived.global_step] = (archive_path, archived)
        elif archived.completed == selected[1].completed and (
            archived.checkpoint_hash != selected[1].checkpoint_hash
        ):
            raise RuntimeError("multiple verified archives map to one Task 7 import destination")

    restored = []
    local_records = load_checkpoint_records(stage)
    local_hashes = {record.checkpoint_hash for record in local_records}
    for step, (archive_path, archived) in sorted(selected_by_step.items()):
        if archived.checkpoint_hash in local_hashes:
            restored.append(archived.checkpoint_hash)
            continue
        if any(record.global_step == step for record in local_records):
            raise FileExistsError("local checkpoint conflicts with a verified archive destination")
        command = [
            "feature-dynamics",
            "rlmf-import-checkpoint",
            "--artifact-root",
            str(ARTIFACT_ROOT),
            "--archive",
            str(archive_path),
            "--config",
            CONFIG_PATH,
            "--stage",
            "pre-sft" if stage["record_stage"] == "pre_sft" else "rl",
        ]
        if stage["record_stage"] == "rl":
            command.extend(
                [
                    "--arm",
                    "standard" if stage["record_arm"] == "standard_grpo" else "rlmf",
                    "--seed",
                    str(stage["seed"]),
                    "--pre-sft-parent-hash",
                    expected_pre_sft_hash,
                ]
            )
        subprocess.run(command, check=True)
        restored.append(archived.checkpoint_hash)
        local_hashes.add(archived.checkpoint_hash)
    return restored


## Run pre-SFT, standard, then RLMF

The CLI owns all model and scientific behavior. Existing bound archives are restored first. During each active CLI process, newly sealed checkpoints are detected and exported immediately; a final scan covers checkpoints sealed as the process exits. Existing archives are verified and skipped, never overwritten.


In [ ]:
PILOT_STEPS = 25
TRAINING_STAGES = [
    {
        "name": "pre_sft",
        "record_stage": "pre_sft",
        "record_arm": None,
        "seed": None,
        "cli": [
            "feature-dynamics",
            "rlmf-train",
            "--config",
            CONFIG_PATH,
            "--artifact-root",
            str(ARTIFACT_ROOT),
            "--stage",
            "pre-sft",
        ],
    },
    {
        "name": "standard",
        "record_stage": "rl",
        "record_arm": "standard_grpo",
        "seed": SELECTED_SEED,
        "cli": [
            "feature-dynamics",
            "rlmf-train",
            "--config",
            CONFIG_PATH,
            "--artifact-root",
            str(ARTIFACT_ROOT),
            "--stage",
            "rl",
            "--arm",
            "standard",
            "--seed",
            str(SELECTED_SEED),
        ],
    },
    {
        "name": "rlmf",
        "record_stage": "rl",
        "record_arm": "rlmf",
        "seed": SELECTED_SEED,
        "cli": [
            "feature-dynamics",
            "rlmf-train",
            "--config",
            CONFIG_PATH,
            "--artifact-root",
            str(ARTIFACT_ROOT),
            "--stage",
            "rl",
            "--arm",
            "rlmf",
            "--seed",
            str(SELECTED_SEED),
        ],
    },
]
if RUN_MODE == "pilot":
    for stage in TRAINING_STAGES[1:]:
        stage["cli"].extend(
            ["--stop-after-step", str(PILOT_STEPS), "--infrastructure-pilot"]
        )

checkpoint_summary = {}
published_archives = {}
pre_sft_parent_hash = None
for stage in TRAINING_STAGES:
    restore_stage_archives(stage, pre_sft_parent_hash)
    matching = load_checkpoint_records(stage)
    for sealed_record in matching:
        published_archives[sealed_record.checkpoint_hash] = publish_checkpoint_archive(
            sealed_record, stage, pre_sft_parent_hash
        )

    if RUN_MODE == "pilot" and stage["record_stage"] == "rl":
        if any(record.completed for record in matching):
            raise RuntimeError("infrastructure pilot cannot reuse a completed confirmatory arm")
        stage_ready = any(
            not record.completed and record.global_step == PILOT_STEPS for record in matching
        )
    else:
        stage_ready = any(record.completed for record in matching)

    started_with_checkpoint = bool(matching)
    if not stage_ready:
        command = list(stage["cli"])
        if matching:
            command.append("--resume")
        process = subprocess.Popen(command)
        while process.poll() is None:
            for sealed_record in load_checkpoint_records(stage):
                if sealed_record.checkpoint_hash not in published_archives:
                    published_archives[sealed_record.checkpoint_hash] = publish_checkpoint_archive(
                        sealed_record, stage, pre_sft_parent_hash
                    )
            time.sleep(1)
        for sealed_record in load_checkpoint_records(stage):
            if sealed_record.checkpoint_hash not in published_archives:
                published_archives[sealed_record.checkpoint_hash] = publish_checkpoint_archive(
                    sealed_record, stage, pre_sft_parent_hash
                )
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, command)

    matching = load_checkpoint_records(stage)
    if RUN_MODE == "pilot":
        if stage["record_stage"] == "rl":
            if any(record.completed for record in matching):
                raise RuntimeError("pilot RL checkpoint must remain incomplete")
            pilot_records = [
                record
                for record in matching
                if not record.completed and record.global_step == PILOT_STEPS
            ]
            if not pilot_records:
                raise RuntimeError("infrastructure pilot did not seal the incomplete step-25 artifact")
            record = max(pilot_records, key=lambda item: (item.global_step, item.micro_step))
            if record.completed or record.global_step != PILOT_STEPS:
                raise RuntimeError("invalid infrastructure pilot completion state")
            stage_status = "infrastructure_pilot_incomplete"
        else:
            completed_records = [record for record in matching if record.completed]
            if len(completed_records) != 1:
                raise RuntimeError("pre-SFT must have one completed canonical checkpoint")
            record = completed_records[0]
            stage_status = "verified_existing" if stage_ready else "completed"
    else:
        completed_records = [record for record in matching if record.completed]
        if len(completed_records) != 1:
            raise RuntimeError(f"stage requires one completed checkpoint: {stage['name']}")
        record = completed_records[0]
        stage_status = (
            "verified_existing"
            if stage_ready
            else ("resumed" if started_with_checkpoint else "completed")
        )

    if record.checkpoint_hash not in published_archives:
        published_archives[record.checkpoint_hash] = publish_checkpoint_archive(
            record, stage, pre_sft_parent_hash
        )
    stage_archives = [
        published_archives[item.checkpoint_hash]
        for item in matching
        if item.checkpoint_hash in published_archives
    ]
    checkpoint_summary[stage["name"]] = {
        "archive_count": len(stage_archives),
        "archives": stage_archives,
        "checkpoint_hash": record.checkpoint_hash,
        "completed": record.completed,
        "global_step": record.global_step,
        "status": stage_status,
    }
    if stage["record_stage"] == "pre_sft":
        pre_sft_parent_hash = record.checkpoint_hash

run_state = (
    "infrastructure_pilot_incomplete"
    if RUN_MODE == "pilot"
    else "checkpoint_sequence_complete"
)
print(json.dumps({"checkpoints": checkpoint_summary, "run_state": run_state}, sort_keys=True))


## Evaluation boundary

Task 10 has not supplied executable rollout commands or the locked validation-audit/test ordering boundary. This notebook therefore exposes no evaluation switch and records the state as unavailable.


In [ ]:
EVALUATION_STATE = {
    "owner": "task_10",
    "status": "not_available",
}
print(json.dumps({"evaluation": EVALUATION_STATE}, sort_keys=True))


In [ ]:
completion_summary = {
    "artifact_root": str(ARTIFACT_ROOT),
    "checkpoints": checkpoint_summary,
    "config": CONFIG_PATH,
    "drive_enabled": USE_DRIVE,
    "evaluation": EVALUATION_STATE,
    "project_commit": resolved_commit,
    "run_mode": RUN_MODE,
    "run_state": run_state,
    "schema_version": 1,
    "seed": SELECTED_SEED,
}
print(json.dumps(completion_summary, sort_keys=True))
